# Plot Checkpoint Predictions - v2

Select a trained masked-event checkpoint, one session date, and a ticker list. The notebook rebuilds chronological event-chunk windows, runs inference through the forecast head, and plots predicted future mid vs target future mid for each ticker.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v2"
MODEL_ROOT = Path(r"D:\TradingData\quant-research-workbench\market_data\models\masked_event_model") / MODEL_VERSION

# Paste a trained v2 checkpoint path, or leave empty to auto-load the latest v2 checkpoint_last.pt.
CHECKPOINT_PATH = ""

# Inference selection.
WORKSPACE = None  # Leave None to auto-detect local repo or workstation runtime root.
CACHE_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\event_chunks_v2")
CANONICAL_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\canonical_events_v2")
SESSION_DATE = "2025-11-03"
TICKERS = ("AAPL", "MSFT", "NVDA")
HORIZON_INDEX = 0  # 0 means the first available target horizon.

# Runtime controls.
DEVICE = "cuda"
USE_AMP = True
INFERENCE_BATCH_SIZE = 512

# 0 means plot every valid chronological window for each ticker.
MAX_POINTS_PER_TICKER = 0

ARCHITECTURE_OUTPUT_DIR = MODEL_ROOT / "plot_architecture"

In [ ]:
import sys
from dataclasses import fields
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader


def resolve_workspace(explicit):
    if explicit:
        path = Path(explicit)
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path(r"D:\TradingCodes\quant-research-workbench"),
        Path(r"D:\TradingCodes\quant-research-workbench-masked-event-v2-runtime"),
    ])
    for path in candidates:
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    raise FileNotFoundError("Could not find a workspace root containing research/masked_event_model/v2.")


WORKSPACE = resolve_workspace(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v2.config import DataConfig, ExperimentConfig, LossConfig, MaskConfig, ModelConfig, TrainConfig
from research.masked_event_model.v2.data import EventChunkDataset, discover_chunk_files, parse_tickers, target_horizons_from_columns
from research.masked_event_model.v2.masking import MaskBatch
from research.masked_event_model.v2.model import MaskedEventAutoencoder
from research.masked_event_model.v2.model_artifacts import save_model_architecture_artifacts
from research.masked_event_model.v2.schema import CHUNK_SUMMARY_COLUMNS, QUOTE_FEATURE_COLUMNS, TRADE_FEATURE_COLUMNS
from research.masked_event_model.v2.targets import decode_binary_magnitude_logits_to_bps

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (18, 8),
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "legend.fontsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "lines.linewidth": 2.0,
    }
)

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)
torch.set_float32_matmul_precision("high")

In [ ]:
def dataclass_from_dict(cls, payload, tuple_fields=()):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in payload.items() if key in allowed}
    for name in ("cache_root", "canonical_root", "output_root"):
        if name in values:
            values[name] = Path(values[name])
    for name in tuple_fields:
        if name in values:
            values[name] = tuple(values[name])
    return cls(**values)


def latest_checkpoint(model_root):
    candidates = list(model_root.glob("*/checkpoint_last.pt")) + list(model_root.glob("*/last.pt")) + list(model_root.glob("*/best.pt"))
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found under {model_root}. Set CHECKPOINT_PATH manually.")
    return max(candidates, key=lambda path: path.stat().st_mtime)


checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(MODEL_ROOT)
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
if not TICKERS:
    raise ValueError("Set TICKERS to at least one ticker symbol.")

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location="cpu")
checkpoint_config = checkpoint.get("config")
if not checkpoint_config:
    raise KeyError("Checkpoint does not contain a config payload, so model/data shape cannot be restored safely.")
checkpoint_version = checkpoint_config.get("experiment_version") or checkpoint_config.get("version")
if checkpoint_version and checkpoint_version != MODEL_VERSION:
    raise ValueError(f"Checkpoint version {checkpoint_version!r} does not match notebook version {MODEL_VERSION!r}.")

data_config = dataclass_from_dict(DataConfig, checkpoint_config.get("data", {}), tuple_fields=("tickers",))
model_config = dataclass_from_dict(ModelConfig, checkpoint_config.get("model", {}))
mask_config = dataclass_from_dict(MaskConfig, checkpoint_config.get("masks", {}))
loss_config = dataclass_from_dict(LossConfig, checkpoint_config.get("losses", {}))
train_config = dataclass_from_dict(TrainConfig, checkpoint_config.get("train", {}))

# Keep checkpoint model/data shape, but override the plotted session/tickers/runtime.
data_config.cache_root = CACHE_ROOT
data_config.canonical_root = CANONICAL_ROOT
data_config.train_start_date = SESSION_DATE
data_config.train_end_date = SESSION_DATE
data_config.validation_start_date = SESSION_DATE
data_config.validation_end_date = SESSION_DATE
data_config.test_start_date = SESSION_DATE
data_config.test_end_date = SESSION_DATE
data_config.tickers = parse_tickers(TICKERS)
data_config.max_files = 0
data_config.max_windows_per_file = 0
data_config.shuffle_files = False
data_config.shuffle_windows = False
train_config.batch_size = INFERENCE_BATCH_SIZE
train_config.amp = USE_AMP

config = ExperimentConfig(data=data_config, masks=mask_config, model=model_config, losses=loss_config, train=train_config)
files = discover_chunk_files(config.data, start_date=SESSION_DATE, end_date=SESSION_DATE, tickers=config.data.tickers)
if not files:
    raise FileNotFoundError(f"No chunk files found under {config.data.cache_root} for {SESSION_DATE} and tickers={config.data.tickers}")
schema = pl.scan_parquet(str(files[0].path)).collect_schema()
target_horizons = target_horizons_from_columns(schema.names())
if not target_horizons:
    raise ValueError(f"No target horizon columns found in {files[0].path}")
if HORIZON_INDEX < 0 or HORIZON_INDEX >= len(target_horizons):
    raise ValueError(f"HORIZON_INDEX={HORIZON_INDEX} outside available horizons {target_horizons}")
selected_horizon = int(target_horizons[HORIZON_INDEX])

print(f"workspace={WORKSPACE}")
print(f"checkpoint={checkpoint_path}")
print(f"checkpoint_step={checkpoint.get('step')} best_score={checkpoint.get('best_score')}")
print(f"device={device} amp={config.train.amp and device.type == 'cuda'} batch_size={config.train.batch_size}")
print(f"session={SESSION_DATE} tickers={config.data.tickers}")
print(f"context_seconds={config.data.context_seconds} chunk_ms={config.data.chunk_ms} context_chunks={config.data.context_chunks}")
print(f"available target_horizons={target_horizons} selected={selected_horizon}")
print(f"features quote={len(QUOTE_FEATURE_COLUMNS)} trade={len(TRADE_FEATURE_COLUMNS)} summary={len(CHUNK_SUMMARY_COLUMNS)}")

In [ ]:
model = MaskedEventAutoencoder(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS),
    trade_feature_count=len(TRADE_FEATURE_COLUMNS),
    summary_feature_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=config.data.context_chunks,
    max_quote_events=config.data.max_quote_events,
    max_trade_events=config.data.max_trade_events,
    max_total_events=config.data.max_total_events,
    horizon_count=len(target_horizons),
    target_bit_count=config.data.target_bit_count,
    config=config.model,
).to(device)
model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

param_count = sum(parameter.numel() for parameter in model.parameters())
print(f"loaded model parameters={param_count:,}")

In [ ]:
# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

from IPython.display import Markdown, Image, display

SUMMARY_BATCH_SIZE = 1
SUMMARY_DEPTH = 8
GRAPH_DEPTH = 3

ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
architecture_info = save_model_architecture_artifacts(
    model=model,
    data_config=config.data,
    output_dir=ARCHITECTURE_OUTPUT_DIR,
    version=f"mem-{MODEL_VERSION}-plot",
    torch_module=torch,
    wandb_run=None,
    summary_batch_size=SUMMARY_BATCH_SIZE,
    summary_depth=SUMMARY_DEPTH,
    graph_depth=GRAPH_DEPTH,
)

display(Markdown(f"### {MODEL_VERSION} Model Summary"))
print("architecture_dir:", architecture_info.get("architecture_dir"))
print("Forward input shapes:", architecture_info.get("input_shapes"))
print("summary:", architecture_info.get("summary_path"))
print("graph png:", architecture_info.get("graph_png_path"))
print("graph svg:", architecture_info.get("graph_svg_path"))
print("graph pdf:", architecture_info.get("graph_pdf_path"))
print("errors:", architecture_info.get("errors"))

summary_path = Path(architecture_info["summary_path"])
if summary_path.exists():
    text = summary_path.read_text(encoding="utf-8")
    display(Markdown("```text\n" + text[:30000] + ("\n... truncated ..." if len(text) > 30000 else "") + "\n```"))

png_path = architecture_info.get("graph_png_path")
if png_path and Path(png_path).exists():
    display(Image(filename=str(png_path)))

In [ ]:
def empty_masks_for(batch_dict):
    return MaskBatch(
        quote_value_mask=torch.zeros_like(batch_dict["quote_values"], dtype=torch.bool),
        trade_value_mask=torch.zeros_like(batch_dict["trade_values"], dtype=torch.bool),
        summary_value_mask=torch.zeros_like(batch_dict["chunk_summary"], dtype=torch.bool),
        event_kind_mask=torch.zeros_like(batch_dict["event_kinds"], dtype=torch.bool),
        quote_token_mask=torch.zeros_like(batch_dict["quote_values"][..., 0], dtype=torch.bool),
        trade_token_mask=torch.zeros_like(batch_dict["trade_values"][..., 0], dtype=torch.bool),
        chunk_mask=torch.zeros_like(batch_dict["chunk_summary"][..., 0], dtype=torch.bool),
    )


def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}


def timestamp_from_ns(value):
    return datetime.fromtimestamp(int(value) / 1_000_000_000, tz=timezone.utc).replace(tzinfo=None)


def infer_session_timeline_predictions(model, config, target_horizons, selected_horizon_index, device, max_points_per_ticker=0):
    dataset = EventChunkDataset(config=config.data, split="train", batch_size=config.train.batch_size, seed=17)
    loader = DataLoader(dataset, batch_size=None, num_workers=0)
    rows_by_ticker = {ticker: [] for ticker in config.data.tickers}
    selected_horizon = int(target_horizons[selected_horizon_index])
    max_points = int(max_points_per_ticker or 0)
    with torch.inference_mode():
        for batch in loader:
            device_batch = move_tensor_batch(batch, device)
            masks = empty_masks_for(device_batch)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=config.train.amp and device.type == "cuda"):
                output = model(
                    device_batch["quote_values"],
                    device_batch["trade_values"],
                    device_batch["event_kinds"],
                    device_batch["event_indices"],
                    device_batch["chunk_summary"],
                    masks,
                )
            pred_bps = decode_binary_magnitude_logits_to_bps(output.forecast_logits.detach().cpu().numpy()).reshape(batch["target_bps"].shape)
            target_bps = batch["target_bps"].numpy()
            current_mid = batch["current_mid"].numpy()
            origin_ns = batch["origin_timestamp_ns"].numpy()
            tickers = batch["ticker"]
            for idx, ticker in enumerate(tickers):
                if ticker not in rows_by_ticker:
                    rows_by_ticker[ticker] = []
                if max_points and len(rows_by_ticker[ticker]) >= max_points:
                    continue
                pred = float(pred_bps[idx, selected_horizon_index, 0])
                target = float(target_bps[idx, selected_horizon_index, 0])
                current = float(current_mid[idx])
                target_time_ns = int(origin_ns[idx]) + selected_horizon * config.data.chunk_ms * 1_000_000
                rows_by_ticker[ticker].append(
                    {
                        "ticker": ticker,
                        "origin_timestamp_ns": int(origin_ns[idx]),
                        "origin_time": timestamp_from_ns(origin_ns[idx]).isoformat(),
                        "target_timestamp_ns": target_time_ns,
                        "target_time": timestamp_from_ns(target_time_ns).isoformat(),
                        "horizon_chunks": selected_horizon,
                        "current_mid": current,
                        "target_mid": float(current * np.exp(target / 10000.0)),
                        "prediction_mid": float(current * np.exp(pred / 10000.0)),
                        "target_bps": target,
                        "prediction_bps": pred,
                        "abs_error_bps": abs(pred - target),
                        "dir_correct": bool((pred > 0.0) == (target > 0.0)),
                    }
                )
    return {ticker: sorted(rows, key=lambda row: int(row["origin_timestamp_ns"])) for ticker, rows in rows_by_ticker.items() if rows}


rows_by_ticker = infer_session_timeline_predictions(
    model=model,
    config=config,
    target_horizons=target_horizons,
    selected_horizon_index=HORIZON_INDEX,
    device=device,
    max_points_per_ticker=MAX_POINTS_PER_TICKER,
)

if not rows_by_ticker:
    raise RuntimeError("No valid prediction rows were created. Check date, tickers, context length, chunk cache, and horizons.")

summary_rows = []
for ticker, rows in rows_by_ticker.items():
    target_bps_values = np.array([float(row["target_bps"]) for row in rows])
    prediction_bps_values = np.array([float(row["prediction_bps"]) for row in rows])
    target_mid_values = np.array([float(row["target_mid"]) for row in rows])
    prediction_mid_values = np.array([float(row["prediction_mid"]) for row in rows])
    error_bps = prediction_bps_values - target_bps_values
    direction_correct = (prediction_bps_values > 0.0) == (target_bps_values > 0.0)
    summary_rows.append(
        {
            "ticker": ticker,
            "windows": len(rows),
            "target_start": rows[0]["target_time"],
            "target_end": rows[-1]["target_time"],
            "mae_price": float(np.mean(np.abs(prediction_mid_values - target_mid_values))),
            "mae_bps": float(np.mean(np.abs(error_bps))),
            "rmse_bps": float(np.sqrt(np.mean(error_bps ** 2))),
            "bias_bps": float(np.mean(error_bps)),
            "direction_accuracy_pct": float(np.mean(direction_correct) * 100.0),
        }
    )

summary = pl.DataFrame(summary_rows)
summary

In [ ]:
def plot_ticker_predictions(ticker, rows):
    rows = sorted(rows, key=lambda row: int(row["origin_timestamp_ns"]))
    times = [datetime.fromisoformat(str(row["target_time"])) for row in rows]
    target_mid = np.array([float(row["target_mid"]) for row in rows])
    prediction_mid = np.array([float(row["prediction_mid"]) for row in rows])
    target_bps = np.array([float(row["target_bps"]) for row in rows])
    prediction_bps = np.array([float(row["prediction_bps"]) for row in rows])
    error_bps = prediction_bps - target_bps
    direction_correct = (prediction_bps > 0.0) == (target_bps > 0.0)

    mae_bps = float(np.mean(np.abs(error_bps)))
    rmse_bps = float(np.sqrt(np.mean(error_bps ** 2)))
    bias_bps = float(np.mean(error_bps))
    dir_acc = float(np.mean(direction_correct) * 100.0)

    fig, (price_ax, error_ax) = plt.subplots(
        2,
        1,
        figsize=(18, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.1], "hspace": 0.08},
    )
    fig.suptitle(
        f"{ticker} | {SESSION_DATE} | h{selected_horizon} mid prediction vs target | "
        f"MAE {mae_bps:.2f} bps | RMSE {rmse_bps:.2f} bps | bias {bias_bps:.2f} bps | dir {dir_acc:.1f}%",
        y=0.98,
        fontweight="bold",
    )

    price_ax.plot(times, target_mid, color="#111827", label="target mid", linewidth=2.3)
    price_ax.plot(times, prediction_mid, color="#f97416c7", label="prediction mid", linewidth=1.0)
    price_ax.set_ylabel("mid price")
    price_ax.legend(loc="upper left", frameon=True)
    price_ax.grid(True, color="#e5e7eb", linewidth=0.8)

    error_ax.axhline(0.0, color="#374151", linewidth=1.0)
    error_ax.fill_between(times, error_bps, 0.0, color="#f97316", alpha=0.18)
    error_ax.plot(times, error_bps, color="#ea580c", linewidth=1.3, label="prediction error bps")
    error_ax.set_ylabel("error bps")
    error_ax.set_xlabel("target chunk time")
    error_ax.grid(True, color="#e5e7eb", linewidth=0.8)

    locator = mdates.AutoDateLocator(minticks=5, maxticks=12)
    formatter = mdates.ConciseDateFormatter(locator)
    error_ax.xaxis.set_major_locator(locator)
    error_ax.xaxis.set_major_formatter(formatter)
    fig.autofmt_xdate()
    plt.show()


for ticker, rows in rows_by_ticker.items():
    plot_ticker_predictions(ticker, rows)

In [ ]:
# Optional: inspect the exact plotted rows as a table.
all_rows = []
for ticker, rows in rows_by_ticker.items():
    for row in rows:
        all_rows.append(row)

pl.DataFrame(all_rows).select(
    "ticker",
    "origin_time",
    "target_time",
    "horizon_chunks",
    "current_mid",
    "target_mid",
    "prediction_mid",
    "target_bps",
    "prediction_bps",
    "abs_error_bps",
    "dir_correct",
).head(50)